# 10 - OASIS Black-box Shadow Model MIA

Notebook autonome pour tester une attaque MIA de type shadow-model en mode black-box sur la data OASIS preparee.

Objectifs:
- Charger prepared_oasis2_transformer.npz
- Entrainer un modele cible tabulaire cohérent
- Introduire un leger overfitting sans fuite de donnees
- Evaluer une attaque shadow-model standard
- Comparer shadow_meta avec les baselines loss et logistic

In [ ]:
import os
import sys
import random
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, roc_auc_score
from tensorflow.keras import layers, Model

repo_root = os.getcwd()
proto_dir = os.path.join(repo_root, 'transformer_pipeline')
if proto_dir not in sys.path:
    sys.path.append(proto_dir)

import research_protocol as rp
importlib.reload(rp)
evaluate_mia_research_protocol = rp.evaluate_mia_research_protocol
set_seed = rp.set_seed

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

def set_global_determinism(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

In [ ]:
SEED = 42

# Target model - force clear overfitting without leakage
TARGET_DROPOUT = 0.00
TARGET_EPOCHS = 900
TARGET_BATCH_SIZE = 4
TARGET_LR = 2e-3
TARGET_MEMBER_FRACTION = 0.10

# Data allocation (same as before)
TARGET_EXTRA_FROM_SHADOW_FRACTION = 0.35
TARGET_TEST_SIZE = 0.35
SHADOW_FROM_BASE_TEST_FRACTION = 0.10

# Shadow-model attack settings - aligned to target regime
N_SHADOWS = 100
SHADOW_EPOCHS = 220
SHADOW_BATCH_SIZE = 8
SHADOW_LR = 2e-3
SHADOW_MEMBER_FRACTION = TARGET_MEMBER_FRACTION

ATTACK_MIN_AUC_TARGET = 0.60

set_global_determinism(SEED)

candidate_paths = [
    os.path.join('data_preparation', 'prepared_oasis2_transformer.npz'),
    os.path.join('..', 'data_preparation', 'prepared_oasis2_transformer.npz'),
]
prepared_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if prepared_path is None:
    raise FileNotFoundError('prepared_oasis2_transformer.npz introuvable. Executer le notebook de data prep OASIS.')

b = np.load(prepared_path)
X_train_base = b['X_train'].astype(np.float32)
X_test_base = b['X_test'].astype(np.float32)
y_train_base = b['y_train'].astype(np.int32)
y_test_base = b['y_test'].astype(np.int32)

if 'X_shadow' in b:
    X_shadow_base = b['X_shadow'].astype(np.float32)
else:
    X_shadow_base = b['X_shadow_raw'].astype(np.float32)
y_shadow_base = b['y_shadow'].astype(np.int32)

base_target_n = len(X_train_base) + len(X_test_base)
base_shadow_n = len(X_shadow_base)

rng = np.random.default_rng(SEED + 1234)

# 1) Move a small disjoint slice from base test to shadow pool.
test_to_shadow_n = int(len(X_test_base) * SHADOW_FROM_BASE_TEST_FRACTION)
test_to_shadow_idx = rng.choice(len(X_test_base), size=test_to_shadow_n, replace=False)
test_keep_mask = np.ones(len(X_test_base), dtype=bool)
test_keep_mask[test_to_shadow_idx] = False

X_test_for_target = X_test_base[test_keep_mask]
y_test_for_target = y_test_base[test_keep_mask]

X_shadow_aug = np.vstack([X_shadow_base, X_test_base[test_to_shadow_idx]])
y_shadow_aug = np.concatenate([y_shadow_base, y_test_base[test_to_shadow_idx]])

# 2) Move part of augmented shadow to target pool (same logic as best run).
extra_n = int(len(X_shadow_aug) * TARGET_EXTRA_FROM_SHADOW_FRACTION)
extra_idx = rng.choice(len(X_shadow_aug), size=extra_n, replace=False)
keep_mask = np.ones(len(X_shadow_aug), dtype=bool)
keep_mask[extra_idx] = False

X_target_pool = np.vstack([X_train_base, X_test_for_target, X_shadow_aug[extra_idx]])
y_target_pool = np.concatenate([y_train_base, y_test_for_target, y_shadow_aug[extra_idx]])

X_shadow_raw = X_shadow_aug[keep_mask]
y_shadow = y_shadow_aug[keep_mask]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_target_pool,
    y_target_pool,
    test_size=TARGET_TEST_SIZE,
    random_state=SEED,
    stratify=y_target_pool,
)

X_train_seq = X_train.reshape(-1, 1, X_train.shape[1]).astype(np.float32)
X_test_seq = X_test.reshape(-1, 1, X_test.shape[1]).astype(np.float32)

# Build target member subset (strict subset of train split only)
rng_target = np.random.default_rng(SEED + 2026)
n_target_members = max(32, int(len(X_train_seq) * TARGET_MEMBER_FRACTION))
target_member_idx = rng_target.choice(len(X_train_seq), size=n_target_members, replace=False)
X_train_target_seq = X_train_seq[target_member_idx]
y_train_target = y_train[target_member_idx]

print(f'Prepared file: {prepared_path}')
print(f'Base sizes: target={base_target_n}, shadow={base_shadow_n}')
print(f'Test moved to shadow: {test_to_shadow_n}')
print(f'New sizes: train={len(X_train_seq)}, test={len(X_test_seq)}, shadow={len(X_shadow_raw)}')
print(f'Target member subset: {len(X_train_target_seq)} / {len(X_train_seq)} ({TARGET_MEMBER_FRACTION:.0%})')
print(f'Class ratio train={y_train.mean():.4f}, test={y_test.mean():.4f}, shadow={y_shadow.mean():.4f}')

def build_tabular_mlp(n_features, dropout=0.15, width1=512, width2=256, width3=128, lr=2e-3):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Flatten()(inp)
    x = layers.Dense(width1, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(width2, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(width3, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

Prepared file: data_preparation\prepared_oasis2_transformer.npz
Base sizes: target=354, shadow=107
Test moved to shadow: 28
New sizes: train=242, test=131, shadow=88
Target member subset: 72 / 242 (30%)
Class ratio train=0.3430, test=0.3435, shadow=0.4205


In [3]:
print('=== Target model (aggressive overfitting for membership signal) ===')
set_seed(SEED + 2)
tf.keras.backend.clear_session()

model_attack = build_tabular_mlp(
    n_features=X_train.shape[1],
    dropout=TARGET_DROPOUT,
    width1=512,
    width2=256,
    width3=128,
    lr=TARGET_LR,
)
model_attack.fit(
    X_train_target_seq,
    y_train_target,
    epochs=TARGET_EPOCHS,
    batch_size=TARGET_BATCH_SIZE,
    verbose=0,
)

p_tr_attack_member = model_attack.predict(X_train_target_seq, verbose=0).ravel()
p_tr_attack_full = model_attack.predict(X_train_seq, verbose=0).ravel()
p_te_attack = model_attack.predict(X_test_seq, verbose=0).ravel()

attack_train_member_acc = accuracy_score(y_train_target, (p_tr_attack_member >= 0.5).astype(int))
attack_train_full_acc = accuracy_score(y_train, (p_tr_attack_full >= 0.5).astype(int))
attack_test_acc = accuracy_score(y_test, (p_te_attack >= 0.5).astype(int))

display(pd.DataFrame([{
    'model': 'attack_target',
    'train_acc_member_subset': attack_train_member_acc,
    'train_acc_full_pool': attack_train_full_acc,
    'test_acc': attack_test_acc,
    'gap_member_minus_test': attack_train_member_acc - attack_test_acc,
    'train_auc_member_subset': roc_auc_score(y_train_target, p_tr_attack_member),
    'test_auc': roc_auc_score(y_test, p_te_attack),
}]).round(4))

print(f'✓ Target config: {TARGET_EPOCHS} epochs, batch={TARGET_BATCH_SIZE}, lr={TARGET_LR}')
print(f'✓ Architecture: 512->256->128, dropout={TARGET_DROPOUT}, member_fraction={TARGET_MEMBER_FRACTION}')

=== Target model (aggressive overfitting for membership signal) ===



,model,train_acc_member_subset,train_acc_full_pool,test_acc,gap_member_minus_test,train_auc_member_subset,test_auc
0,attack_target,1.0,0.8926,0.9313,0.0687,1.0,0.9437


✓ Target config: 900 epochs, batch=4, lr=0.002
✓ Architecture: 512->256->128, dropout=0.0, member_fraction=0.3


In [5]:
print('=== Shadow-model black-box attack (reinforced + aligned) ===')
import inspect

extra_kwargs = {}
sig = inspect.signature(evaluate_mia_research_protocol)
if 'optimize_low_fpr_threshold' in sig.parameters:
    extra_kwargs['optimize_low_fpr_threshold'] = True
if 'max_fpr_target' in sig.parameters:
    extra_kwargs['max_fpr_target'] = 0.05
if 'shadow_member_fraction' in sig.parameters:
    extra_kwargs['shadow_member_fraction'] = SHADOW_MEMBER_FRACTION

baseline_summary, baseline_per_seed = evaluate_mia_research_protocol(
    target_model=model_attack,
    X_train_seq=X_train_target_seq,
    y_train=y_train_target,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_tabular_mlp(n_features=nf, dropout=0.15, width1=512, width2=256, width3=128, lr=SHADOW_LR),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH_SIZE,
    **extra_kwargs,
)
display(baseline_summary.round(4))

quick_view = baseline_summary[['attack', 'auc_mean', 'auc_std', 'tpr_at_1pct_fpr_mean', 'tpr_at_5pct_fpr_mean']].copy()
display(quick_view.round(4))

shadow_auc = float(baseline_summary.loc[baseline_summary['attack'] == 'shadow_meta', 'auc_mean'].iloc[0])
shadow_tpr_5 = float(baseline_summary.loc[baseline_summary['attack'] == 'shadow_meta', 'tpr_at_5pct_fpr_mean'].iloc[0])

if shadow_auc < ATTACK_MIN_AUC_TARGET:
    print(f'⚠️  WARNING: shadow_meta={shadow_auc:.4f} (below target {ATTACK_MIN_AUC_TARGET:.2f})')
else:
    print(f'✓ GOOD: shadow_meta AUC={shadow_auc:.4f} | TPR@5% FPR={shadow_tpr_5:.4f}')

print(f'✓ Attack config: {N_SHADOWS} shadows × {SHADOW_EPOCHS} epochs, batch={SHADOW_BATCH_SIZE}, lr={SHADOW_LR}, member_fraction={SHADOW_MEMBER_FRACTION}')

=== Shadow-model black-box attack (reinforced + aligned) ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,tpr_at_1pct_fpr_mean,tpr_at_1pct_fpr_std,tpr_at_5pct_fpr_mean,tpr_at_5pct_fpr_std
1,shadow_meta,0.5975,0.0220,0.6463,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0
2,threshold_loss,0.5868,0.0148,0.4683,0.0185,0.3985,0.0085,0.9862,0.0189,0.5675,0.0095,0.0,0.0,0.0,0.0
0,logistic,0.5571,0.0376,0.6463,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0


,attack,auc_mean,auc_std,tpr_at_1pct_fpr_mean,tpr_at_5pct_fpr_mean
1,shadow_meta,0.5975,0.0220,0.0,0.0
2,threshold_loss,0.5868,0.0148,0.0,0.0
0,logistic,0.5571,0.0376,0.0,0.0


⚠️  WARNING: shadow_meta=0.5975 (below target 0.60)
✓ Attack config: 80 shadows × 220 epochs, batch=8, lr=0.002, member_fraction=0.3
